# Stable Diffusion v1.5 + LoRA (comparison)

LoRA fine-tuning of Stable Diffusion v1.5 on the flowers at 256x256 with class prompts, a guidance sweep, and the same evaluation after downsampling to 32x32.

In [1]:
!pip install -q torch-fidelity diffusers transformers accelerate

In [2]:
import os; os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
MODEL_ID   = "stable-diffusion-v1-5/stable-diffusion-v1-5"
VARIANT    = "fp16"
DTYPE_NAME = "float16"
RES        = 256
LORA_RANK  = 32
LR         = 1e-4
BATCH      = 16
TRAIN_STEPS = 3000
SAVE_EVERY = 1000
CAPTION_DROP = 0.1
PROMPT     = "a photo of a {} flower"
SAMPLE_STEPS = 25
GUIDANCE_SWEEP = [1.0, 1.5, 2.0]
SWEEP_N    = 2000
FINAL_N    = 10000
GEN_BS     = 64
SEED       = 0

In [3]:
import os, sys, glob, json, math, time, copy
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np, torch, torch.nn.functional as F, torchvision, kagglehub
import torchvision.transforms as T
from torchvision.utils import save_image
from tqdm.auto import tqdm
from diffusers import UNet2DConditionModel, AutoencoderKL, DDPMScheduler, DPMSolverMultistepScheduler
from transformers import CLIPTextModel, CLIPTokenizer
for m in [k for k in list(sys.modules) if k.startswith("torchmetrics")]:
    del sys.modules[m]
import torch_fidelity
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.kid import KernelInceptionDistance
from torchmetrics.image.inception import InceptionScore

assert torch.cuda.is_available(), "NO GPU! Settings -> Accelerator -> GPU T4 x2"
DEVS = [torch.device(f"cuda:{i}") for i in range(torch.cuda.device_count())]
DEV = DEVS[0]; DTYPE = getattr(torch, DTYPE_NAME)
OUT = "/kaggle/working/sd_lora"; os.makedirs(OUT, exist_ok=True)
print("GPUs:", [torch.cuda.get_device_name(d) for d in DEVS])
torch.manual_seed(SEED)

hits = sorted(glob.glob("/kaggle/input/**/flowers-recognition", recursive=True))
path = hits[0] if hits else kagglehub.dataset_download("alxmamaev/flowers-recognition")
ds = torchvision.datasets.ImageFolder(root=path)
files = [p for p, _ in ds.samples]
classes = sorted({Path(p).parent.name for p in files}); cls_idx = {c: i for i, c in enumerate(classes)}
labels = torch.tensor([cls_idx[Path(p).parent.name] for p in files])
print("images:", len(files), "| classes:", {c: int((labels == i).sum()) for c, i in cls_idx.items()})
from PIL import Image
to_t = T.ToTensor()
rs_hi = T.Resize((RES, RES), interpolation=T.InterpolationMode.BICUBIC, antialias=True)
rs_32 = T.Resize((32, 32), interpolation=T.InterpolationMode.BICUBIC, antialias=True)
hi, lo = [], []
for p in tqdm(files, desc="load"):
    x = to_t(Image.open(p).convert("RGB")); hi.append(rs_hi(x).clamp(0, 1)); lo.append(rs_32(x))
X_hi = torch.stack(hi); del hi
real_u8 = (torch.stack(lo).clamp(0, 1) * 255).round().to(torch.uint8); del lo
print("train:", tuple(X_hi.shape), "| real ref:", tuple(real_u8.shape))

kw = dict(torch_dtype=DTYPE, **({"variant": VARIANT} if VARIANT else {}))
tokenizer = CLIPTokenizer.from_pretrained(MODEL_ID, subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(MODEL_ID, subfolder="text_encoder", **kw).to(DEV).eval()
vae = AutoencoderKL.from_pretrained(MODEL_ID, subfolder="vae", **kw).to(DEV).eval()
unet = UNet2DConditionModel.from_pretrained(MODEL_ID, subfolder="unet", **kw).to(DEV)
noise_sched = DDPMScheduler.from_pretrained(MODEL_ID, subfolder="scheduler")
SF = vae.config.scaling_factor
print(f"unet params: {sum(p.numel() for p in unet.parameters())/1e6:.0f}M")

@torch.no_grad()
def embed(texts):
    ids = tokenizer(texts, padding="max_length", max_length=tokenizer.model_max_length, truncation=True, return_tensors="pt").input_ids
    return text_encoder(ids.to(DEV))[0]
prompts = [PROMPT.format(c) for c in classes] + [""]
EMB = embed(prompts)
print("prompts:", prompts)

@torch.no_grad()
def encode(x):
    lat = []
    for i in range(0, len(x), 32):
        xb = (x[i:i + 32] * 2 - 1).to(DEV, DTYPE)
        lat.append((vae.encode(xb).latent_dist.mean * SF).float().cpu())
    return torch.cat(lat)
LAT = torch.stack([encode(X_hi), encode(X_hi.flip(3))], 1)
print("latents:", tuple(LAT.shape))

GPUs: ['Tesla T4', 'Tesla T4']
images: 4317 | classes: {'daisy': 764, 'dandelion': 1052, 'rose': 784, 'sunflower': 733, 'tulip': 984}


train: (4317, 3, 256, 256) | real ref: (4317, 3, 32, 32)


unet params: 860M
prompts: ['a photo of a daisy flower', 'a photo of a dandelion flower', 'a photo of a rose flower', 'a photo of a sunflower flower', 'a photo of a tulip flower', '']
latents: (4317, 2, 4, 32, 32)


In [4]:
class LoRALinear(torch.nn.Module):
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.scale = base, alpha / r
        self.lora_A = torch.nn.Linear(base.in_features, r, bias=False, device=base.weight.device)
        self.lora_B = torch.nn.Linear(r, base.out_features, bias=False, device=base.weight.device)
        torch.nn.init.normal_(self.lora_A.weight, std=1 / r); torch.nn.init.zeros_(self.lora_B.weight)
    def forward(self, x, *args, **kwargs):
        lora = self.lora_B(self.lora_A(x.to(self.lora_A.weight.dtype))) * self.scale
        return self.base(x) + lora.to(x.dtype)

def add_lora(model, r, alpha):
    targets = []
    for mod in model.modules():
        for attr in ("to_q", "to_k", "to_v"):
            if isinstance(getattr(mod, attr, None), torch.nn.Linear): targets.append((mod, attr))
        if isinstance(getattr(mod, "to_out", None), torch.nn.ModuleList) and isinstance(mod.to_out[0], torch.nn.Linear):
            targets.append((mod.to_out, 0))
    for parent, key in targets:
        if isinstance(key, int): parent[key] = LoRALinear(parent[key], r, alpha)
        else: setattr(parent, key, LoRALinear(getattr(parent, key), r, alpha))
    return len(targets)

unet.requires_grad_(False)
print("LoRA layers added:", add_lora(unet, LORA_RANK, LORA_RANK))
params = [p for p in unet.parameters() if p.requires_grad]
print(f"trainable LoRA params: {sum(p.numel() for p in params)/1e6:.2f}M")
opt = torch.optim.AdamW(params, lr=LR, weight_decay=1e-2)
scaler = torch.amp.GradScaler("cuda", enabled=(DTYPE == torch.float16))
LAT_d, LAB_d = LAT.to(DEV), labels.to(DEV)
N = len(LAT_d); log = []; t0 = time.time(); unet.train()
for step in tqdm(range(1, TRAIN_STEPS + 1), desc="train"):
    idx = torch.randint(0, N, (BATCH,), device=DEV)
    flip = torch.randint(0, 2, (BATCH,), device=DEV)
    z0 = LAT_d[idx, flip]
    y = torch.where(torch.rand(BATCH, device=DEV) < CAPTION_DROP, torch.full_like(LAB_d[idx], len(classes)), LAB_d[idx])
    noise = torch.randn_like(z0)
    t = torch.randint(0, noise_sched.config.num_train_timesteps, (BATCH,), device=DEV)
    zt = noise_sched.add_noise(z0, noise, t)
    with torch.autocast("cuda", dtype=DTYPE, enabled=(DTYPE == torch.float16)):
        pred = unet(zt.to(DTYPE), t, encoder_hidden_states=EMB[y]).sample
    loss = F.mse_loss(pred.float(), noise)
    opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
    scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, 1.0)
    scaler.step(opt); scaler.update()
    if step % 100 == 0 or step == 1:
        log.append(dict(step=step, loss=loss.item(), minutes=round((time.time() - t0) / 60, 2))); print(log[-1])
    if step % SAVE_EVERY == 0:
        torch.save({n: p.detach().cpu() for n, p in unet.named_parameters() if p.requires_grad}, f"{OUT}/lora_step{step}.pt")
train_minutes = (time.time() - t0) / 60
json.dump(log, open(f"{OUT}/train_log.json", "w"), indent=2)
unet.eval(); print(f"training done in {train_minutes:.1f} min")
del opt, scaler, LAT_d, LAB_d; import gc; gc.collect(); torch.cuda.empty_cache()

LoRA layers added: 128
trainable LoRA params: 6.38M


{'step': 1, 'loss': 0.25310927629470825, 'minutes': 0.02}
{'step': 100, 'loss': 0.2874683141708374, 'minutes': 1.23}
{'step': 200, 'loss': 0.2717698812484741, 'minutes': 2.45}
{'step': 300, 'loss': 0.16915616393089294, 'minutes': 3.66}
{'step': 400, 'loss': 0.14303377270698547, 'minutes': 4.88}
{'step': 500, 'loss': 0.15860068798065186, 'minutes': 6.11}
{'step': 600, 'loss': 0.19535139203071594, 'minutes': 7.33}
{'step': 700, 'loss': 0.22507056593894958, 'minutes': 8.56}
{'step': 800, 'loss': 0.13080927729606628, 'minutes': 9.79}
{'step': 900, 'loss': 0.1689976304769516, 'minutes': 11.01}
{'step': 1000, 'loss': 0.21191099286079407, 'minutes': 12.24}
{'step': 1100, 'loss': 0.14935141801834106, 'minutes': 13.47}
{'step': 1200, 'loss': 0.19755148887634277, 'minutes': 14.69}
{'step': 1300, 'loss': 0.1141490787267685, 'minutes': 15.92}
{'step': 1400, 'loss': 0.1826329380273819, 'minutes': 17.14}
{'step': 1500, 'loss': 0.27064642310142517, 'minutes': 18.36}
{'step': 1600, 'loss': 0.150524884

In [5]:
unets, vaes, embs = [unet], [vae], [EMB]
for d in DEVS[1:]:
    unets.append(copy.deepcopy(unet).to(d).eval()); vaes.append(copy.deepcopy(vae).to(d).eval()); embs.append(EMB.to(d))
class_p = torch.bincount(labels, minlength=len(classes)).float(); class_p /= class_p.sum()

@torch.no_grad()
def gen_on(i, n, seed, g_scale, tag=None):
    d = DEVS[i]; un, va, em = unets[i], vaes[i], embs[i]
    gen = torch.Generator(device=d).manual_seed(seed * 1000 + i); outs, hires = [], []
    for s in tqdm(range(0, n, GEN_BS), desc=f"gpu{i} g={g_scale}", position=i, leave=False):
        b = min(GEN_BS, n - s)
        y = torch.multinomial(class_p.to(d), b, replacement=True, generator=gen)
        sch = DPMSolverMultistepScheduler.from_config(noise_sched.config); sch.set_timesteps(SAMPLE_STEPS, device=d)
        lh, lw = LAT.shape[-2:]
        z = torch.randn((b, 4, lh, lw), device=d, generator=gen) * sch.init_noise_sigma
        for t in sch.timesteps:
            zin = sch.scale_model_input(z, t)
            with torch.autocast("cuda", dtype=DTYPE, enabled=(DTYPE == torch.float16)):
                if g_scale == 1.0:
                    eps = un(zin.to(DTYPE), t, encoder_hidden_states=em[y]).sample.float()
                else:
                    e = un(torch.cat([zin, zin]).to(DTYPE), t, encoder_hidden_states=torch.cat([em[y], em[[len(classes)] * b]])).sample.float()
                    eps = e[b:] + g_scale * (e[:b] - e[b:])
            z = sch.step(eps, t, z).prev_sample
        img = torch.cat([va.decode((z[j:j + 16] / SF).to(DTYPE)).sample.float()
                         for j in range(0, b, 16)]).clamp(-1, 1)
        img = (img + 1) / 2
        if not hires: hires.append(img[:16].cpu())
        lo = F.interpolate(img, size=(32, 32), mode="bicubic", antialias=True, align_corners=False).clamp(0, 1)
        outs.append((lo * 255).round().to(torch.uint8).cpu())
        if tag: np.save(f"{OUT}/{tag}_gpu{i}.npy", torch.cat(outs).numpy())
    return torch.cat(outs), hires[0]

def generate(n, seed, g_scale, tag=None):
    k = len(DEVS); shares = [n // k + (1 if i < n % k else 0) for i in range(k)]
    with ThreadPoolExecutor(k) as ex:
        res = list(ex.map(lambda i: gen_on(i, shares[i], seed, g_scale, tag), range(k)))
    return torch.cat([r[0] for r in res]), torch.cat([r[1] for r in res])

@torch.no_grad()
def fid_kid(a_u8, b_u8):
    torch.cuda.empty_cache()
    fid = FrechetInceptionDistance(feature=2048).to(DEV)
    kid = KernelInceptionDistance(subset_size=min(1000, len(a_u8), len(b_u8))).to(DEV)
    for x, flag in ((a_u8, True), (b_u8, False)):
        for i in range(0, len(x), 100):
            xb = x[i:i + 100].to(DEV); fid.update(xb, real=flag); kid.update(xb, real=flag)
    km, ks = kid.compute(); out = (fid.compute().item(), km.item(), ks.item())
    del fid, kid; torch.cuda.empty_cache()
    return out

t1 = time.time(); g_small, h_small = generate(2 * len(DEVS), SEED + 99, 1.0)
save_image(h_small, f"{OUT}/preview_hires.png", nrow=4); save_image(g_small.float() / 255, f"{OUT}/preview_32.png", nrow=4)
print(f"preview done in {time.time() - t1:.1f}s")

preview done in 3.4s


In [6]:
sweep = []
for g_scale in GUIDANCE_SWEEP:
    t1 = time.time(); g2k, hi2k = generate(SWEEP_N, SEED, g_scale)
    f, km, ks = fid_kid(real_u8, g2k)
    sweep.append(dict(guidance=g_scale, fid_2k=f, kid_2k=km, minutes=round((time.time() - t1) / 60, 1))); print(sweep[-1])
    save_image(hi2k, f"{OUT}/sweep_g{g_scale}_hires.png", nrow=8)
    json.dump(sweep, open(f"{OUT}/sweep.json", "w"), indent=2)
best_g = min(sweep, key=lambda r: r["fid_2k"])["guidance"]; print("best guidance:", best_g)

{'guidance': 1.0, 'fid_2k': 37.42181396484375, 'kid_2k': 0.012482947669923306, 'minutes': 10.5}


{'guidance': 1.5, 'fid_2k': 33.254066467285156, 'kid_2k': 0.008190526627004147, 'minutes': 19.9}


{'guidance': 2.0, 'fid_2k': 34.74269104003906, 'kid_2k': 0.007248756941407919, 'minutes': 20.0}
best guidance: 1.5


In [7]:
t1 = time.time(); gen, hi = generate(FINAL_N, SEED + 1, best_g, tag="final")
final_minutes = (time.time() - t1) / 60
np.save(f"{OUT}/final_samples.npy", gen.numpy())
save_image(gen[:64].float() / 255, f"{OUT}/final_samples_grid.png", nrow=8)
save_image(hi, f"{OUT}/final_samples_hires.png", nrow=8)
res = dict(model=MODEL_ID, res=RES, lora_rank=LORA_RANK, train_steps=TRAIN_STEPS, train_minutes=round(train_minutes, 1),
           sample_steps=SAMPLE_STEPS, guidance=best_g, n_total=len(gen), sample_minutes=round(final_minutes, 1))
r2k = [fid_kid(real_u8, gen[j:j + 2000]) for j in range(0, len(gen) - 1999, 2000)]
res["fid_2k"] = [round(r[0], 3) for r in r2k]; res["kid_2k"] = [r[1] for r in r2k]
res["fid_2k_mean"], res["fid_2k_std"] = float(np.mean(res["fid_2k"])), float(np.std(res["fid_2k"]))
res["kid_2k_mean"], res["kid_2k_std"] = float(np.mean(res["kid_2k"])), float(np.std(res["kid_2k"]))
Ns = [n for n in (2000, 4000, 6000, 8000, 10000) if n <= len(gen)]; fid_at = {}
for n in Ns:
    vals = [fid_kid(real_u8, gen[torch.randperm(len(gen), generator=torch.Generator().manual_seed(r))[:n]])[0]
            for r in range(3 if n < len(gen) else 1)]
    fid_at[n] = float(np.mean(vals)); print(f"FID@{n}: {fid_at[n]:.3f}")
res["fid_at_N"] = fid_at
if len(Ns) >= 2:
    b, a = np.polyfit([1 / n for n in Ns], [fid_at[n] for n in Ns], 1); res["fid_inf"], res["fid_inf_slope"] = float(a), float(b)
json.dump(res, open(f"{OUT}/final_results.json", "w"), indent=2); print("core results saved")

class U8(torch.utils.data.Dataset):
    def __init__(self, x): self.x = x
    def __len__(self): return len(self.x)
    def __getitem__(self, i): return self.x[i]
try:
    m = InceptionScore().to(DEV)
    for i in range(0, len(gen), 100): m.update(gen[i:i + 100].to(DEV))
    mu, sd = m.compute(); res["is_mean"], res["is_std"] = mu.item(), sd.item()
except Exception as e: res["is_error"] = repr(e); print("IS failed:", e)
try:
    r = torch_fidelity.calculate_metrics(input1=U8(gen[:2000]), input2=U8(real_u8), prc=True, cuda=True, batch_size=100, verbose=False)
    res["precision"], res["recall"] = r["precision"], r["recall"]
except Exception as e: res["prc_error"] = repr(e); print("precision/recall failed:", e)
R = real_u8.float().flatten(1) / 255; G = gen[:64].float().flatten(1) / 255; exact = "donot_use_mm_for_euclid_dist"
d_gen, idx = torch.cdist(G, R, compute_mode=exact).min(1)
q = torch.randperm(len(R), generator=torch.Generator().manual_seed(0))[:64]
dr = torch.cdist(R[q], R, compute_mode=exact); dr[torch.arange(64), q] = float("inf")
res["nn_dist_gen"], res["nn_dist_real"] = d_gen.mean().item(), dr.min(1).values.mean().item()
save_image(torch.stack([gen[:64], real_u8[idx]], 1).flatten(0, 1)[:32].float() / 255, f"{OUT}/final_nearest_neighbours.png", nrow=8)
json.dump(res, open(f"{OUT}/final_results.json", "w"), indent=2)
print(json.dumps(res, indent=2))

FID@2000: 33.652
FID@4000: 26.910
FID@6000: 24.798
FID@8000: 23.729
FID@10000: 23.167
core results saved


{
  "model": "stable-diffusion-v1-5/stable-diffusion-v1-5",
  "res": 256,
  "lora_rank": 32,
  "train_steps": 3000,
  "train_minutes": 36.7,
  "sample_steps": 25,
  "guidance": 1.5,
  "n_total": 10000,
  "sample_minutes": 95.8,
  "fid_2k": [
    33.519,
    32.95,
    33.457,
    33.138,
    32.539
  ],
  "kid_2k": [
    0.008497204631567001,
    0.008688957430422306,
    0.008371264673769474,
    0.008294043131172657,
    0.007781631778925657
  ],
  "fid_2k_mean": 33.1206,
  "fid_2k_std": 0.35764597019958055,
  "kid_2k_mean": 0.00832662032917142,
  "kid_2k_std": 0.0003034646900051802,
  "fid_at_N": {
    "2000": 33.651588439941406,
    "4000": 26.909595489501953,
    "6000": 24.797971725463867,
    "8000": 23.728883743286133,
    "10000": 23.167470932006836
  },
  "fid_inf": 20.438644226400502,
  "fid_inf_slope": 26331.932144406688,
  "is_mean": 5.837800979614258,
  "is_std": 0.1673973947763443,
  "precision": 0.7074999809265137,
  "recall": 0.5566365718841553,
  "nn_dist_gen": 11.296